In [0]:
#Recreate the employees DataFrame and yesterday's profile_column() function
from pyspark.sql.functions import col, when

data = [(1, "Alice", "IT", 55000),
        (2, "Ben", "HR", None),
        (3, "Cara", "IT", 62000),
        (4, "Dev", None, 45000),
        (5, "Alice", "IT", 55000)]
columns = ["id", "name", "department", "salary"]
employees = spark.createDataFrame(data, columns)

def profile_column(df, column_name):
    total = df.count()
    non_null = df.filter(col(column_name).isNotNull()).count()
    distinct = df.select(column_name).distinct().count()
    return {
        "column": column_name,
        "total_rows": total,
        "null_pct": round((total - non_null) / total * 100, 1),
        "distinct_count": distinct,
    }

In [0]:
#Collect results and build the report DataFrame
report_rows = [profile_column(employees, c) for c in employees.columns]
report_df = spark.createDataFrame(report_rows)
report_df.display()


In [0]:
# Sort by the worst offenders
report_sorted = report_df.orderBy(col("null_pct").desc())
report_sorted.display()

In [0]:
# Add the status flag
report_flagged = report_sorted.withColumn(
    "status",
    when(col("null_pct") > 20, "FAIL")
    .when(col("null_pct") > 5, "WARNING")
    .otherwise("OK")
)
report_flagged.display()

In [0]:
# Save as Delta Table
report_flagged.write.format("delta").mode("overwrite").saveAsTable("workspace.default.data_profile_report")

spark.sql("SELECT * FROM workspace.default.data_profile_report ORDER BY null_pct DESC").display()